In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os

BASE = "/content/drive/MyDrive/Deep Learning/CVPR Research Paper Final"
DATA = os.path.join(BASE, "data")
PRO = os.path.join(BASE, "processed")

os.makedirs(DATA, exist_ok=True)
os.makedirs(PRO, exist_ok=True)

print("BASE:", BASE)
print("DATA:", DATA)
print("PRO:", PRO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/Deep Learning/CVPR Research Paper Final
DATA: /content/drive/MyDrive/Deep Learning/CVPR Research Paper Final/data
PRO: /content/drive/MyDrive/Deep Learning/CVPR Research Paper Final/processed


In [2]:
!pip -q install openneuro-py mne pandas numpy scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 22.5 MB/s eta 0:00:00


In [4]:
import os

DS_PATH = os.path.join(DATA, "ds004504")

print("Target:", DS_PATH)

if not os.path.exists(DS_PATH):
    os.makedirs(DS_PATH, exist_ok=True)

print("Downloading... (if it stops, run this cell again)")

!openneuro-py download --dataset=ds004504 --target-dir="{DS_PATH}" --include=participants.tsv --include=derivatives

print("Done")


Streaming output truncated to the last 5000 lines.

sub-033_task-eyesclosed_eeg.set:   8% 2.26M/28.7M [00:02<00:18, 1.53MB/s]
sub-035_task-eyesclosed_eeg.set:   0% 0.00/30.1M [00:00<?, ?B/s]


sub-034_task-eyesclosed_eeg.set:   9% 3.39M/39.4M [00:02<00:18, 2.04MB/s]



sub-033_task-eyesclosed_eeg.set:   9% 2.49M/28.7M [00:02<00:16, 1.63MB/s]
sub-031_task-eyesclosed_eeg.set:  18% 8.47M/46.7M [00:03<00:09, 4.34MB/s]

sub-035_task-eyesclosed_eeg.set:   0% 32.5k/30.1M [00:00<02:11, 239kB/s]


sub-034_task-eyesclosed_eeg.set:   9% 3.67M/39.4M [00:02<00:17, 2.08MB/s]
sub-031_task-eyesclosed_eeg.set:  19% 9.01M/46.7M [00:03<00:08, 4.64MB/s]



sub-033_task-eyesclosed_eeg.set:   9% 2.72M/28.7M [00:02<00:16, 1.67MB/s]

sub-036_task-eyesclosed_eeg.set:   0% 32.5k/34.2M [00:00<02:14, 266kB/s]


sub-035_task-eyesclosed_eeg.set:   0% 83.5k/30.1M [00:00<01:33, 336kB/s]
sub-031_task-eyesclosed_eeg.set:  21% 9.58M/46.7M [00:04<00:07, 4.97MB/s]



sub-033_task-eyesclosed_eeg.set:  10% 2.97M/28.7M [00:0

In [6]:
import pandas as pd

df = pd.read_csv(participants_file, sep="\t")

print(df["Group"].value_counts(dropna=False))
print("Unique:", sorted(df["Group"].dropna().astype(str).unique())[:20])


Group
A    36
C    29
F    23
Name: count, dtype: int64
Unique: ['A', 'C', 'F']


In [7]:
import os
import pandas as pd

participants_file = os.path.join(DS_PATH, "participants.tsv")
df = pd.read_csv(participants_file, sep="\t")

df["participant_id"] = df["participant_id"].astype(str).str.strip()
df["Group"] = df["Group"].astype(str).str.strip()

df2 = df[df["Group"].isin(["A", "C"])].copy()

# A = AD, C = CN
df2["label"] = df2["Group"].map({"C": 0, "A": 1})

print("Counts:")
print(df2["Group"].value_counts())

out_csv = os.path.join(PRO, "labels_ad_cn.csv")
df2[["participant_id", "Group", "label"]].to_csv(out_csv, index=False)
print("Saved:", out_csv)


Counts:
Group
A    36
C    29
Name: count, dtype: int64
Saved: /content/drive/MyDrive/Deep Learning/CVPR Research Paper Final/processed/labels_ad_cn.csv


In [8]:
import os, pandas as pd

lab_df = pd.read_csv(os.path.join(PRO, "labels_ad_cn.csv"))
sub0 = lab_df["participant_id"].iloc[0]

p1 = os.path.join(DS_PATH, "derivatives", sub0, "eeg")
print("Example subject:", sub0)
print("EEG folder exists?", os.path.exists(p1))

if os.path.exists(p1):
    print("SET files:", [f for f in os.listdir(p1) if f.endswith(".set")][:5])


Example subject: sub-001
EEG folder exists? True
SET files: ['sub-001_task-eyesclosed_eeg.set']


In [9]:
import os

print("DS_PATH:", DS_PATH)
print("participants exists:", os.path.exists(os.path.join(DS_PATH, "participants.tsv")))
print("derivatives exists:", os.path.exists(os.path.join(DS_PATH, "derivatives")))


DS_PATH: /content/drive/MyDrive/Deep Learning/CVPR Research Paper Final/data/ds004504
participants exists: True
derivatives exists: True


In [10]:
import os
import numpy as np
import pandas as pd
import mne
import tensorflow as tf
from scipy.signal import welch

mne.set_log_level("ERROR")

labels_file = os.path.join(PRO, "labels_ad_cn.csv")
lab_df = pd.read_csv(labels_file)

want_ch = ["Fp1","Fp2","F7","F3","Fz","F4","F8","T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]

epoch_sec = 30
epochs_per_sub = 20
fmin, fmax = 4.0, 40.0
bins = 128

X_list = []
y_list = []
subj_list = []
skipped = []

def get_set_path(sub_id):
    p = os.path.join(DS_PATH, "derivatives", sub_id, "eeg")
    if not os.path.exists(p):
        return None
    fs = [f for f in os.listdir(p) if f.endswith(".set")]
    if len(fs) == 0:
        return None
    fs.sort()
    return os.path.join(p, fs[0])

target_f = np.linspace(fmin, fmax, bins)

for i in range(len(lab_df)):
    sub_id = str(lab_df.loc[i, "participant_id"]).strip()
    lab = int(lab_df.loc[i, "label"])

    set_path = get_set_path(sub_id)
    if set_path is None:
        skipped.append((sub_id, "no_set"))
        continue

    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
    except:
        skipped.append((sub_id, "read_fail"))
        continue

    try:
        picked = [c for c in want_ch if c in raw.ch_names]
        if len(picked) >= 19:
            raw.pick_channels(picked[:19])
        else:
            eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
            if len(eeg_picks) < 19:
                skipped.append((sub_id, "not_19_eeg"))
                continue
            raw.pick(eeg_picks[:19])

        if int(raw.info["sfreq"]) != 500:
            raw.resample(500)

        data = raw.get_data()
        sf = float(raw.info["sfreq"])

        n_per = int(epoch_sec * sf)
        total_epochs = data.shape[1] // n_per
        use_epochs = min(total_epochs, epochs_per_sub)

        if use_epochs <= 0:
            skipped.append((sub_id, "too_short"))
            continue

        for e in range(use_epochs):
            a = e * n_per
            b = a + n_per
            ep = data[:, a:b]

            freqs, psd = welch(ep, fs=sf, nperseg=int(sf*2), axis=1)

            mask = (freqs >= fmin) & (freqs <= fmax)
            f_sel = freqs[mask]
            p_sel = psd[:, mask]

            out = np.zeros((p_sel.shape[0], bins), dtype=np.float32)
            for ch in range(p_sel.shape[0]):
                out[ch] = np.interp(target_f, f_sel, p_sel[ch]).astype(np.float32)

            mn = out.min()
            mx = out.max()
            if mx > mn:
                out = (out - mn) / (mx - mn)
            else:
                out = np.zeros_like(out)

            out = out[..., None]
            out = tf.image.resize(out, (128, 128)).numpy().astype(np.float32)

            X_list.append(out)
            y_list.append(lab)
            subj_list.append(sub_id)

    except:
        skipped.append((sub_id, "proc_fail"))
        continue

    if (i + 1) % 10 == 0:
        print("subjects done:", i + 1)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
subj = np.array(subj_list)

print("X shape:", X.shape)
print("y counts:", np.unique(y, return_counts=True))
print("skipped:", len(skipped))

np.save(os.path.join(PRO, "X.npy"), X)
np.save(os.path.join(PRO, "y.npy"), y)
np.save(os.path.join(PRO, "subj.npy"), subj)

with open(os.path.join(PRO, "skipped_subjects.txt"), "w") as f:
    for s in skipped:
        f.write(str(s) + "\n")

print("Saved: X.npy, y.npy, subj.npy, skipped_subjects.txt")


/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose

subjects done: 10


/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose

subjects done: 20


/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose

...

subjects done: 50


/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose

subjects done: 60


/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/tmp/ipython-input-1228393613.py:47: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose

X shape: (1287, 128, 128, 1)
y counts: (array([0, 1], dtype=int32), array([580, 707]))
skipped: 0
Saved: X.npy, y.npy, subj.npy, skipped_subjects.txt
